# 17c MEL/NV cue reliance cross evaluation

This notebook checks whether the fixed center synthetic cue is causally useful for the trained binary MEL/NV models.

It evaluates the same checkpoints on:

1. clean MEL/NV CSV, no synthetic cue
2. cued MEL/NV CSV, MEL images contain the green cue

The key quantity is the paired image-level change:

`p(MEL | cued image) - p(MEL | clean image)`

For MEL images, a large positive value means the cue increases MEL confidence.
For NV images, there should be no cue, so clean and cued CSVs should be nearly identical.


In [1]:
from pathlib import Path
import json
import subprocess
import shlex
import sys

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, roc_auc_score

QUAL_SEED = 42

REPO_ROOT = Path("..").resolve()
HAM_ROOT = REPO_ROOT / "data" / "HAM10000"

# ---------------------------------------------------------------------
# Adjust these two paths if your bigger cue folder name differs.
# ---------------------------------------------------------------------
SYN_ROOT = HAM_ROOT / "synthetic_cue" / f"mel_nv_fixed_center_seed{QUAL_SEED}"
CLEAN_CSV = SYN_ROOT / "csv" / "ham_mel_nv_clean.csv"
CUE_CSV = SYN_ROOT / "csv" / f"ham_mel_nv_cue_on_mel_fixed_center_seed{QUAL_SEED}.csv"

OUT_ROOT = REPO_ROOT / "outputs" / f"cue_reliance_cross_eval_seed{QUAL_SEED}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT:", REPO_ROOT)
print("CLEAN_CSV:", CLEAN_CSV, CLEAN_CSV.exists())
print("CUE_CSV:", CUE_CSV, CUE_CSV.exists())
print("OUT_ROOT:", OUT_ROOT)


REPO_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis
CLEAN_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue/mel_nv_fixed_center_seed42/csv/ham_mel_nv_clean.csv True
CUE_CSV: /Users/choekyelnyungmartsang/Developer/master-thesis/data/HAM10000/synthetic_cue/mel_nv_fixed_center_seed42/csv/ham_mel_nv_cue_on_mel_fixed_center_seed42.csv True
OUT_ROOT: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42


## 1. Check CSV pairing

This verifies that the clean and cued CSVs have the same test image IDs. For MEL, clean and cue paths should differ. For NV, they may be identical because no cue was applied.


In [2]:
clean_df = pd.read_csv(CLEAN_CSV)
cue_df = pd.read_csv(CUE_CSV)

clean_test = clean_df[clean_df["split"] == "test"].copy()
cue_test = cue_df[cue_df["split"] == "test"].copy()

print("Clean test counts:")
print(clean_test["gt_label"].value_counts())
print("Cue test counts:")
print(cue_test["gt_label"].value_counts())

merged_paths = clean_test[["image_id", "gt_label", "image_rel_path"]].merge(
    cue_test[["image_id", "gt_label", "image_rel_path", "cue_applied", "cue_mask_rel_path"]],
    on="image_id",
    suffixes=("_clean", "_cue"),
    how="inner",
)

print("Paired rows:", len(merged_paths))
print("Clean-only IDs:", len(set(clean_test["image_id"]) - set(cue_test["image_id"])))
print("Cue-only IDs:", len(set(cue_test["image_id"]) - set(clean_test["image_id"])))
print("Cue applied by GT:")
print(merged_paths.groupby(["gt_label_clean", "cue_applied"]).size())

display(merged_paths.head(10))


Clean test counts:
gt_label
MEL    70
NV     70
Name: count, dtype: int64
Cue test counts:
gt_label
MEL    70
NV     70
Name: count, dtype: int64
Paired rows: 140
Clean-only IDs: 0
Cue-only IDs: 0
Cue applied by GT:
gt_label_clean  cue_applied
MEL             True           70
NV              False          70
dtype: int64


,image_id,gt_label_clean,image_rel_path_clean,gt_label_cue,image_rel_path_cue,cue_applied,cue_mask_rel_path
0,ISIC_0024459,MEL,images/ISIC_0024459.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
1,ISIC_0024571,MEL,images/ISIC_0024571.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
2,ISIC_0024624,MEL,images/ISIC_0024624.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
3,ISIC_0024640,MEL,images/ISIC_0024640.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
4,ISIC_0024756,MEL,images/ISIC_0024756.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
5,ISIC_0024886,MEL,images/ISIC_0024886.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
6,ISIC_0024967,MEL,images/ISIC_0024967.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
7,ISIC_0025105,MEL,images/ISIC_0025105.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
8,ISIC_0025132,MEL,images/ISIC_0025132.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...
9,ISIC_0025234,MEL,images/ISIC_0025234.jpg,MEL,synthetic_cue/mel_nv_fixed_center_seed42/image...,True,synthetic_cue/mel_nv_fixed_center_seed42/cue_m...


## 2. Define checkpoints

Use the clean CE checkpoint and the bigger cue CE / bigger cue HA checkpoints.

Adjust the paths if your copied checkpoint filenames differ.


In [3]:
SIZE = "small"
CHECKPOINTS = {
    "Clean CE": REPO_ROOT / "external" / f"checkpoints2_{SIZE}" / "checkpoint-best-clean.pth",
    "Cue CE": REPO_ROOT / "external" / f"checkpoints2_{SIZE}" / "checkpoint-best-cue.pth",
    "Cue HA": REPO_ROOT / "external" / f"checkpoints2_{SIZE}" / "checkpoint-best-cue-ha.pth",
}

for name, path in CHECKPOINTS.items():
    print(name, "->", path, "exists=", path.exists())


Clean CE -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-clean.pth exists= True
Cue CE -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue.pth exists= True
Cue HA -> /Users/choekyelnyungmartsang/Developer/master-thesis/external/checkpoints2_small/checkpoint-best-cue-ha.pth exists= True


## 3. Lightweight local evaluator

This avoids calling the training script and works on CPU. It loads the PanDerm binary classifier and evaluates image-level probabilities.


In [4]:
# Add PanDerm classification code to Python path.
CLASSIFICATION_DIR = REPO_ROOT / "external" / "PanDerm" / "classification"
if str(CLASSIFICATION_DIR) not in sys.path:
    sys.path.insert(0, str(CLASSIFICATION_DIR))

from models.modeling_finetune import panderm_base_patch16_224_finetune

CLASS_NAMES = ["MEL", "NV"]
CLASS_TO_IDX = {"MEL": 0, "NV": 1}

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

mean = [0.485, 0.456, 0.406]
std = [0.228, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])


def resolve_path(root: Path, rel_or_abs: str) -> Path:
    p = Path(str(rel_or_abs))
    if p.is_absolute():
        return p
    return root / p


def load_checkpoint_into_model(checkpoint_path: Path):
    model = panderm_base_patch16_224_finetune(
        pretrained=False,
        num_classes=2,
        drop_rate=0.0,
        drop_path_rate=0.2,
        attn_drop_rate=0.0,
        drop_block_rate=None,
        use_mean_pooling=True,
        init_scale=1.0,
        use_rel_pos_bias=False,
        init_values=0.1,
        lin_probe=False,
    )
    ckpt = torch.load(checkpoint_path, map_location="cpu")
    state = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    # handle possible backbone. prefix from wrappers
    state = state.copy()
    for key in list(state.keys()):
        if key.startswith("backbone."):
            state[key.replace("backbone.", "", 1)] = state.pop(key)
    msg = model.load_state_dict(state, strict=False)
    print(f"Loaded {checkpoint_path.name}: missing={len(msg.missing_keys)} unexpected={len(msg.unexpected_keys)}")
    model.to(DEVICE)
    model.eval()
    return model


@torch.no_grad()
def predict_csv(checkpoint_name: str, checkpoint_path: Path, csv_path: Path, split="test", batch_size=32):
    df = pd.read_csv(csv_path)
    df = df[df["split"] == split].copy().reset_index(drop=True)
    model = load_checkpoint_into_model(checkpoint_path)

    rows = []
    for start in range(0, len(df), batch_size):
        batch_df = df.iloc[start:start + batch_size]
        images = []
        for _, row in batch_df.iterrows():
            img_path = resolve_path(HAM_ROOT, row["image_rel_path"])
            img = Image.open(img_path).convert("RGB")
            images.append(eval_transform(img))
        x = torch.stack(images).to(DEVICE)
        logits = model(x)
        if isinstance(logits, (tuple, list)):
            logits = logits[0]
        elif isinstance(logits, dict):
            logits = logits["logits"]
        probs = F.softmax(logits, dim=1).detach().cpu().numpy()
        pred_idx = probs.argmax(axis=1)

        for i, (_, row) in enumerate(batch_df.iterrows()):
            true_idx = int(row["binary_label"]) if "binary_label" in row else CLASS_TO_IDX[row["gt_label"]]
            rows.append({
                "checkpoint": checkpoint_name,
                "csv_name": csv_path.stem,
                "image_id": row["image_id"],
                "gt_label": row["gt_label"],
                "true_idx": true_idx,
                "pred_idx": int(pred_idx[i]),
                "pred_label": CLASS_NAMES[int(pred_idx[i])],
                "p_mel": float(probs[i, 0]),
                "p_nv": float(probs[i, 1]),
                "image_rel_path": row["image_rel_path"],
                "cue_applied": bool(row.get("cue_applied", False)),
                "cue_mask_rel_path": row.get("cue_mask_rel_path", ""),
                "cue_area_frac_of_lesion": row.get("cue_area_frac_of_lesion", np.nan),
            })
    return pd.DataFrame(rows)


def summarize_predictions(pred_df):
    y_true = pred_df["true_idx"].to_numpy()
    y_pred = pred_df["pred_idx"].to_numpy()
    p_mel = pred_df["p_mel"].to_numpy()
    out = {
        "n": len(pred_df),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }
    try:
        # positive class for sklearn binary AUC should be class 1. Here class 1 is NV.
        out["auc_roc_nv_positive"] = roc_auc_score(y_true, pred_df["p_nv"].to_numpy())
    except Exception:
        out["auc_roc_nv_positive"] = np.nan
    return out


DEVICE: cpu


/Users/choekyelnyungmartsang/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/Users/choekyelnyungmartsang/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


## 4. Run cross evaluation

This evaluates each checkpoint on both CSVs.


In [5]:
all_preds = []
summary_rows = []

EVAL_CSVS = {
    "clean_test": CLEAN_CSV,
    "cued_test": CUE_CSV,
}

for ckpt_name, ckpt_path in CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f"[skip] missing checkpoint: {ckpt_name} {ckpt_path}")
        continue
    for eval_name, csv_path in EVAL_CSVS.items():
        print("Evaluating:", ckpt_name, "on", eval_name)
        pred_df = predict_csv(ckpt_name, ckpt_path, csv_path, split="test", batch_size=32)
        pred_df["eval_set"] = eval_name
        all_preds.append(pred_df)
        s = summarize_predictions(pred_df)
        s.update({"checkpoint": ckpt_name, "eval_set": eval_name})
        summary_rows.append(s)

preds_df = pd.concat(all_preds, axis=0, ignore_index=True)
summary_df = pd.DataFrame(summary_rows)

preds_out = OUT_ROOT / f"cross_eval_predictions_seed{QUAL_SEED}.csv"
summary_out = OUT_ROOT / f"cross_eval_summary_seed{QUAL_SEED}.csv"
preds_df.to_csv(preds_out, index=False)
summary_df.to_csv(summary_out, index=False)

print("Saved:", preds_out)
print("Saved:", summary_out)
display(summary_df[["checkpoint", "eval_set", "n", "balanced_accuracy", "accuracy", "weighted_f1", "recall_macro", "auc_roc_nv_positive"]])


/Users/choekyelnyungmartsang/opt/anaconda3/envs/thesis/lib/python3.11/site-packages/torch/functional.py:507: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /Users/runner/work/_temp/anaconda/conda-bld/pytorch_1711403251597/work/aten/src/ATen/native/TensorShape.cpp:3550.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Evaluating: Clean CE on clean_test
Loaded checkpoint-best-clean.pth: missing=0 unexpected=0
Evaluating: Clean CE on cued_test
Loaded checkpoint-best-clean.pth: missing=0 unexpected=0
Evaluating: Cue CE on clean_test
Loaded checkpoint-best-cue.pth: missing=0 unexpected=0
Evaluating: Cue CE on cued_test
Loaded checkpoint-best-cue.pth: missing=0 unexpected=0
Evaluating: Cue HA on clean_test
Loaded checkpoint-best-cue-ha.pth: missing=0 unexpected=0
Evaluating: Cue HA on cued_test
Loaded checkpoint-best-cue-ha.pth: missing=0 unexpected=0
Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42/cross_eval_predictions_seed42.csv
Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42/cross_eval_summary_seed42.csv


,checkpoint,eval_set,n,balanced_accuracy,accuracy,weighted_f1,recall_macro,auc_roc_nv_positive
0,Clean CE,clean_test,140,0.878571,0.878571,0.877161,0.878571,0.967959
1,Clean CE,cued_test,140,0.800000,0.800000,0.799837,0.800000,0.902245
2,Cue CE,clean_test,140,0.507143,0.507143,0.349013,0.507143,0.892857
3,Cue CE,cued_test,140,0.992857,0.992857,0.992857,0.992857,0.998367
4,Cue HA,clean_test,140,0.500000,0.500000,0.333333,0.500000,0.872041
5,Cue HA,cued_test,140,0.992857,0.992857,0.992857,0.992857,0.997959


## 5. Paired cue reliance analysis

For each checkpoint, pair the same image ID in clean and cued test sets and compute:

`delta_p_mel = p_mel_cued - p_mel_clean`

For MEL images, this should be positive if the cue pushes the model toward MEL.


In [6]:
paired_rows = []

for ckpt_name in preds_df["checkpoint"].unique():
    clean_pred = preds_df[(preds_df["checkpoint"] == ckpt_name) & (preds_df["eval_set"] == "clean_test")].copy()
    cue_pred = preds_df[(preds_df["checkpoint"] == ckpt_name) & (preds_df["eval_set"] == "cued_test")].copy()

    keep_cols = ["image_id", "gt_label", "true_idx", "pred_label", "p_mel", "p_nv", "cue_applied", "cue_area_frac_of_lesion"]
    merged = clean_pred[keep_cols].merge(
        cue_pred[keep_cols],
        on="image_id",
        suffixes=("_clean", "_cued"),
        how="inner",
    )
    merged.insert(0, "checkpoint", ckpt_name)
    merged["delta_p_mel"] = merged["p_mel_cued"] - merged["p_mel_clean"]
    merged["delta_p_nv"] = merged["p_nv_cued"] - merged["p_nv_clean"]
    merged["pred_changed"] = merged["pred_label_clean"] != merged["pred_label_cued"]
    paired_rows.append(merged)

paired_df = pd.concat(paired_rows, axis=0, ignore_index=True)
paired_out = OUT_ROOT / f"paired_clean_vs_cued_delta_seed{QUAL_SEED}.csv"
paired_df.to_csv(paired_out, index=False)
print("Saved:", paired_out)

display(paired_df.head(20))

paired_summary = paired_df.groupby(["checkpoint", "gt_label_clean"]).agg(
    n=("image_id", "count"),
    delta_p_mel_mean=("delta_p_mel", "mean"),
    delta_p_mel_median=("delta_p_mel", "median"),
    delta_p_mel_std=("delta_p_mel", "std"),
    pred_changed_rate=("pred_changed", "mean"),
    p_mel_clean_mean=("p_mel_clean", "mean"),
    p_mel_cued_mean=("p_mel_cued", "mean"),
    cue_area_frac_mean=("cue_area_frac_of_lesion_cued", "mean"),
).reset_index()

summary_delta_out = OUT_ROOT / f"paired_delta_summary_seed{QUAL_SEED}.csv"
paired_summary.to_csv(summary_delta_out, index=False)
print("Saved:", summary_delta_out)
display(paired_summary)


Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42/paired_clean_vs_cued_delta_seed42.csv


,checkpoint,image_id,gt_label_clean,true_idx_clean,pred_label_clean,p_mel_clean,p_nv_clean,cue_applied_clean,cue_area_frac_of_lesion_clean,gt_label_cued,true_idx_cued,pred_label_cued,p_mel_cued,p_nv_cued,cue_applied_cued,cue_area_frac_of_lesion_cued,delta_p_mel,delta_p_nv,pred_changed
0,Clean CE,ISIC_0024459,MEL,0,MEL,0.986659,0.013341,False,NaN,MEL,0,MEL,0.917631,0.082369,True,0.042592,-0.069028,0.069028,False
1,Clean CE,ISIC_0024571,MEL,0,MEL,0.988490,0.011510,False,NaN,MEL,0,MEL,0.954815,0.045185,True,0.043011,-0.033675,0.033675,False
2,Clean CE,ISIC_0024624,MEL,0,MEL,0.930413,0.069587,False,NaN,MEL,0,NV,0.415798,0.584202,True,0.173511,-0.514614,0.514614,True
3,Clean CE,ISIC_0024640,MEL,0,MEL,0.989334,0.010666,False,NaN,MEL,0,MEL,0.984629,0.015371,True,0.040852,-0.004705,0.004705,False
4,Clean CE,ISIC_0024756,MEL,0,MEL,0.952883,0.047117,False,NaN,MEL,0,NV,0.364553,0.635447,True,0.105527,-0.588329,0.588329,True
5,Clean CE,ISIC_0024886,MEL,0,NV,0.484818,0.515183,False,NaN,MEL,0,NV,0.054819,0.945181,True,0.171485,-0.429999,0.429999,False
6,Clean CE,ISIC_0024967,MEL,0,MEL,0.997179,0.002821,False,NaN,MEL,0,MEL,0.991089,0.008911,True,0.110679,-0.006090,0.006090,False
7,Clean CE,ISIC_0025105,MEL,0,MEL,0.995880,0.004120,False,NaN,MEL,0,MEL,0.906589,0.093411,True,0.037703,-0.089291,0.089291,False
8,Clean CE,ISIC_0025132,MEL,0,MEL,0.986213,0.013787,False,NaN,MEL,0,MEL,0.931059,0.068941,True,0.094649,-0.055154,0.055154,False
9,Clean CE,ISIC_0025234,MEL,0,MEL,0.988790,0.011210,False,NaN,MEL,0,MEL,0.989435,0.010565,True,0.059961,0.000645,-0.000645,False


Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42/paired_delta_summary_seed42.csv


,checkpoint,gt_label_clean,n,delta_p_mel_mean,delta_p_mel_median,delta_p_mel_std,pred_changed_rate,p_mel_clean_mean,p_mel_cued_mean,cue_area_frac_mean
0,Clean CE,MEL,70,-0.142613,-0.032819,0.196655,0.157143,0.955530,0.812917,0.084731
1,Clean CE,NV,70,0.000000,0.000000,0.000000,0.000000,0.237628,0.237628,NaN
2,Cue CE,MEL,70,0.928900,0.985361,0.156663,0.971429,0.056145,0.985046,0.084731
3,Cue CE,NV,70,0.000000,0.000000,0.000000,0.000000,0.004516,0.004516,NaN
4,Cue HA,MEL,70,0.969246,0.994989,0.122549,0.985714,0.015261,0.984506,0.084731
5,Cue HA,NV,70,0.000000,0.000000,0.000000,0.000000,0.001655,0.001655,NaN


## 6. Optional: show strongest cue effects

These are the MEL images where adding the cue most increased or decreased p(MEL).


In [7]:
for ckpt_name in paired_df["checkpoint"].unique():
    print("" + "=" * 100)
    print(ckpt_name)
    mel_df = paired_df[(paired_df["checkpoint"] == ckpt_name) & (paired_df["gt_label_clean"] == "MEL")].copy()
    if len(mel_df) == 0:
        continue
    print("Top positive delta_p_mel:")
    display(mel_df.sort_values("delta_p_mel", ascending=False)[[
        "image_id", "p_mel_clean", "p_mel_cued", "delta_p_mel", "pred_label_clean", "pred_label_cued", "cue_area_frac_of_lesion_cued"
    ]].head(10))
    print("Top negative delta_p_mel:")
    display(mel_df.sort_values("delta_p_mel", ascending=True)[[
        "image_id", "p_mel_clean", "p_mel_cued", "delta_p_mel", "pred_label_clean", "pred_label_cued", "cue_area_frac_of_lesion_cued"
    ]].head(10))


Clean CE
Top positive delta_p_mel:


,image_id,p_mel_clean,p_mel_cued,delta_p_mel,pred_label_clean,pred_label_cued,cue_area_frac_of_lesion_cued
65,ISIC_0033299,0.982554,0.985658,0.003104,MEL,MEL,0.036679
58,ISIC_0032182,0.973904,0.976248,0.002344,MEL,MEL,0.043543
9,ISIC_0025234,0.988790,0.989435,0.000645,MEL,MEL,0.059961
24,ISIC_0027179,0.993391,0.993764,0.000373,MEL,MEL,0.043383
34,ISIC_0028965,0.995116,0.995485,0.000369,MEL,MEL,0.043879
68,ISIC_0033885,0.997807,0.997657,-0.000150,MEL,MEL,0.051048
61,ISIC_0032408,0.997963,0.997420,-0.000543,MEL,MEL,0.064866
22,ISIC_0026993,0.998612,0.997515,-0.001097,MEL,MEL,0.103086
38,ISIC_0029454,0.998938,0.997664,-0.001274,MEL,MEL,0.058394
49,ISIC_0030798,0.997447,0.995824,-0.001623,MEL,MEL,0.043577


Top negative delta_p_mel:


,image_id,p_mel_clean,p_mel_cued,delta_p_mel,pred_label_clean,pred_label_cued,cue_area_frac_of_lesion_cued
46,ISIC_0030366,0.800633,0.050827,-0.749807,MEL,NV,0.067444
51,ISIC_0031177,0.769034,0.165787,-0.603247,MEL,NV,0.057676
4,ISIC_0024756,0.952883,0.364553,-0.588329,MEL,NV,0.105527
50,ISIC_0031146,0.755697,0.240443,-0.515253,MEL,NV,0.067938
2,ISIC_0024624,0.930413,0.415798,-0.514614,MEL,NV,0.173511
35,ISIC_0029013,0.683540,0.182714,-0.500826,MEL,NV,0.177942
27,ISIC_0027517,0.986255,0.487423,-0.498833,MEL,NV,0.173705
28,ISIC_0027560,0.949273,0.489715,-0.459559,MEL,NV,0.080254
48,ISIC_0030759,0.805311,0.357764,-0.447547,MEL,NV,0.182852
30,ISIC_0028173,0.803195,0.368963,-0.434231,MEL,NV,0.031534


Cue CE
Top positive delta_p_mel:


...

,image_id,p_mel_clean,p_mel_cued,delta_p_mel,pred_label_clean,pred_label_cued,cue_area_frac_of_lesion_cued
150,ISIC_0025414,0.011816,0.011499,-0.000316,NV,NV,0.014336
149,ISIC_0025234,0.620597,0.999296,0.378699,MEL,MEL,0.059961
198,ISIC_0032182,0.488683,0.998958,0.510275,NV,MEL,0.043543
143,ISIC_0024640,0.288217,0.998965,0.710748,NV,MEL,0.040852
183,ISIC_0030187,0.256352,0.998947,0.742595,NV,MEL,0.037387
208,ISIC_0033885,0.245988,0.999053,0.753064,NV,MEL,0.051048
182,ISIC_0029933,0.206734,0.999243,0.792509,NV,MEL,0.061227
162,ISIC_0026993,0.180971,0.999141,0.818170,NV,MEL,0.103086
164,ISIC_0027179,0.176971,0.999221,0.822250,NV,MEL,0.043383
205,ISIC_0033299,0.149065,0.999237,0.850172,NV,MEL,0.036679


Cue HA
Top positive delta_p_mel:


,image_id,p_mel_clean,p_mel_cued,delta_p_mel,pred_label_clean,pred_label_cued,cue_area_frac_of_lesion_cued
327,ISIC_0030552,0.000869,0.998630,0.997761,NV,MEL,0.179247
326,ISIC_0030366,0.000719,0.998464,0.997745,NV,MEL,0.067444
298,ISIC_0026094,0.001193,0.998843,0.997650,NV,MEL,0.142049
285,ISIC_0024886,0.001225,0.998703,0.997479,NV,MEL,0.171485
343,ISIC_0032462,0.001376,0.998739,0.997362,NV,MEL,0.102054
336,ISIC_0031642,0.001340,0.998689,0.997349,NV,MEL,0.101431
332,ISIC_0031350,0.001579,0.998923,0.997344,NV,MEL,0.180778
293,ISIC_0025748,0.001264,0.998582,0.997318,NV,MEL,0.084185
315,ISIC_0029013,0.000923,0.998147,0.997224,NV,MEL,0.177942
284,ISIC_0024756,0.001474,0.998626,0.997152,NV,MEL,0.105527


Top negative delta_p_mel:


,image_id,p_mel_clean,p_mel_cued,delta_p_mel,pred_label_clean,pred_label_cued,cue_area_frac_of_lesion_cued
290,ISIC_0025414,0.003256,0.003284,0.000028,NV,NV,0.014336
338,ISIC_0032182,0.211598,0.998621,0.787023,NV,MEL,0.043543
289,ISIC_0025234,0.167621,0.999044,0.831422,NV,MEL,0.059961
323,ISIC_0030187,0.084999,0.998581,0.913582,NV,MEL,0.037387
348,ISIC_0033885,0.077172,0.998708,0.921536,NV,MEL,0.051048
283,ISIC_0024640,0.074154,0.998574,0.924420,NV,MEL,0.040852
304,ISIC_0027179,0.040691,0.998939,0.958248,NV,MEL,0.043383
302,ISIC_0026993,0.038477,0.998775,0.960298,NV,MEL,0.103086
322,ISIC_0029933,0.030262,0.998901,0.968639,NV,MEL,0.061227
345,ISIC_0033299,0.027162,0.998913,0.971751,NV,MEL,0.036679


## Interpretation guide

Look mainly at `paired_delta_summary_seed42.csv`.

Important cases:

- If `Big Cue CE` has high `delta_p_mel_mean` for MEL, then the cue is causally pushing predictions toward MEL.
- If `Big Cue CE` has near zero `delta_p_mel_mean`, then the model may not rely strongly on the cue despite high accuracy.
- If performance on `clean_test` stays high for the big cue checkpoints, then natural lesion features remain sufficient.
- If performance drops from `cued_test` to `clean_test`, then the model learned a shortcut.

Clean CSV vs cue CSV is enough for a first cue removal test because the clean CSV contains the original image paths for the same image IDs. A pixel-level cue removal image is only needed later if you want an exact same-file perturbation pipeline or if other preprocessing changed between the clean and cued images.


Clean CE:
p(MEL | clean) = 0.956
p(MEL | cued)  = 0.813
delta          = -0.143
prediction changed rate = 15.7%

Cue CE:
p(MEL | clean) = 0.056
p(MEL | cued)  = 0.985
delta          = +0.929
prediction changed rate = 97.1%

Cue HA:
p(MEL | clean) = 0.015
p(MEL | cued)  = 0.985
delta          = +0.969
prediction changed rate = 98.6%

NV images have no cue in clean or cued CSV.
So p(MEL | clean) == p(MEL | cued).


- So your synthetic cue worked. The model is definitely using the cue causally.
- This means the clean model did not learn the cue, and adding the artificial green square partially disturbs it. That is expected and actually supports the control design.
- FinerCAM/GradCAM may fail to reveal a highly predictive synthetic shortcut, even when the shortcut causally controls the prediction.

In [8]:
# ============================================================
# Robust final cross evaluation summary
# Works for either:
# 1) raw paired rows, or
# 2) already aggregated summary rows
# ============================================================

from pathlib import Path
import pandas as pd

CROSS_ROOT = REPO_ROOT / "outputs" / f"cue_reliance_cross_eval_seed{QUAL_SEED}"

# Try likely output files
candidate_paths = [
    CROSS_ROOT / f"paired_delta_summary_seed{QUAL_SEED}.csv",
    CROSS_ROOT / f"cross_eval_summary_seed{QUAL_SEED}.csv",
    CROSS_ROOT / f"cross_eval_predictions_seed{QUAL_SEED}.csv",
]

existing = [p for p in candidate_paths if p.exists()]

if len(existing) == 0:
    raise FileNotFoundError(
        "Could not find a cross evaluation CSV. Checked:\n"
        + "\n".join(str(p) for p in candidate_paths)
    )

input_path = existing[0]
df = pd.read_csv(input_path)

print("Loaded:", input_path)
print("Columns:")
print(df.columns.tolist())
display(df.head(20))

# Case A: already summarized table
summary_cols = {
    "checkpoint",
    "gt_label_clean",
    "n",
    "delta_p_mel_mean",
    "delta_p_mel_median",
    "delta_p_mel_std",
    "pred_changed_rate",
    "p_mel_clean_mean",
    "p_mel_cued_mean",
    "cue_area_frac_mean",
}

# Case B: raw paired table
raw_cols = {
    "checkpoint",
    "image_id",
    "gt_label_clean",
    "p_mel_clean",
    "p_mel_cued",
    "delta_p_mel",
    "pred_changed",
}

if raw_cols.issubset(set(df.columns)):
    print("Detected raw paired prediction table. Aggregating now.")

    cue_area_col = None
    for candidate in [
        "cue_area_frac_of_lesion_cued",
        "cue_area_frac_of_lesion",
        "cue_area_frac_cued",
    ]:
        if candidate in df.columns:
            cue_area_col = candidate
            break

    agg_dict = {
        "n": ("image_id", "count"),
        "p_mel_clean_mean": ("p_mel_clean", "mean"),
        "p_mel_cued_mean": ("p_mel_cued", "mean"),
        "delta_p_mel_mean": ("delta_p_mel", "mean"),
        "delta_p_mel_median": ("delta_p_mel", "median"),
        "delta_p_mel_std": ("delta_p_mel", "std"),
        "pred_changed_rate": ("pred_changed", "mean"),
    }

    if cue_area_col is not None:
        agg_dict["cue_area_frac_mean"] = (cue_area_col, "mean")

    final_cross_summary = (
        df.groupby(["checkpoint", "gt_label_clean"])
        .agg(**agg_dict)
        .reset_index()
    )

elif {"checkpoint", "gt_label_clean", "n"}.issubset(set(df.columns)):
    print("Detected already aggregated summary table. Using it directly.")
    final_cross_summary = df.copy()

else:
    raise ValueError(
        "Could not detect whether this is a raw paired table or summary table. "
        "Please inspect the displayed columns above."
    )

final_cross_summary_out = CROSS_ROOT / f"final_cross_eval_summary_seed{QUAL_SEED}.csv"
final_cross_summary.to_csv(final_cross_summary_out, index=False)

print("Saved:", final_cross_summary_out)
display(final_cross_summary)

Loaded: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42/paired_delta_summary_seed42.csv
Columns:
['checkpoint', 'gt_label_clean', 'n', 'delta_p_mel_mean', 'delta_p_mel_median', 'delta_p_mel_std', 'pred_changed_rate', 'p_mel_clean_mean', 'p_mel_cued_mean', 'cue_area_frac_mean']


,checkpoint,gt_label_clean,n,delta_p_mel_mean,delta_p_mel_median,delta_p_mel_std,pred_changed_rate,p_mel_clean_mean,p_mel_cued_mean,cue_area_frac_mean
0,Clean CE,MEL,70,-0.142613,-0.032819,0.196655,0.157143,0.955530,0.812917,0.084731
1,Clean CE,NV,70,0.000000,0.000000,0.000000,0.000000,0.237628,0.237628,NaN
2,Cue CE,MEL,70,0.928900,0.985361,0.156663,0.971429,0.056145,0.985046,0.084731
3,Cue CE,NV,70,0.000000,0.000000,0.000000,0.000000,0.004516,0.004516,NaN
4,Cue HA,MEL,70,0.969246,0.994989,0.122549,0.985714,0.015261,0.984506,0.084731
5,Cue HA,NV,70,0.000000,0.000000,0.000000,0.000000,0.001655,0.001655,NaN


Detected already aggregated summary table. Using it directly.
Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42/final_cross_eval_summary_seed42.csv


,checkpoint,gt_label_clean,n,delta_p_mel_mean,delta_p_mel_median,delta_p_mel_std,pred_changed_rate,p_mel_clean_mean,p_mel_cued_mean,cue_area_frac_mean
0,Clean CE,MEL,70,-0.142613,-0.032819,0.196655,0.157143,0.955530,0.812917,0.084731
1,Clean CE,NV,70,0.000000,0.000000,0.000000,0.000000,0.237628,0.237628,NaN
2,Cue CE,MEL,70,0.928900,0.985361,0.156663,0.971429,0.056145,0.985046,0.084731
3,Cue CE,NV,70,0.000000,0.000000,0.000000,0.000000,0.004516,0.004516,NaN
4,Cue HA,MEL,70,0.969246,0.994989,0.122549,0.985714,0.015261,0.984506,0.084731
5,Cue HA,NV,70,0.000000,0.000000,0.000000,0.000000,0.001655,0.001655,NaN


In [9]:
# ============================================================
# MEL-only cue reliance table for thesis/report
# ============================================================

mel_summary = final_cross_summary[
    final_cross_summary["gt_label_clean"] == "MEL"
].copy()

wanted_cols = [
    "checkpoint",
    "n",
    "p_mel_clean_mean",
    "p_mel_cued_mean",
    "delta_p_mel_mean",
    "delta_p_mel_median",
    "delta_p_mel_std",
    "pred_changed_rate",
    "cue_area_frac_mean",
]

available_cols = [c for c in wanted_cols if c in mel_summary.columns]
mel_summary = mel_summary[available_cols].copy()

if "delta_p_mel_mean" in mel_summary.columns:
    mel_summary = mel_summary.sort_values("delta_p_mel_mean", ascending=False)

mel_summary_out = CROSS_ROOT / f"mel_cue_reliance_summary_seed{QUAL_SEED}.csv"
mel_summary.to_csv(mel_summary_out, index=False)

print("MEL cue reliance summary:")
print("Saved:", mel_summary_out)
display(mel_summary)

MEL cue reliance summary:
Saved: /Users/choekyelnyungmartsang/Developer/master-thesis/outputs/cue_reliance_cross_eval_seed42/mel_cue_reliance_summary_seed42.csv


,checkpoint,n,p_mel_clean_mean,p_mel_cued_mean,delta_p_mel_mean,delta_p_mel_median,delta_p_mel_std,pred_changed_rate,cue_area_frac_mean
4,Cue HA,70,0.015261,0.984506,0.969246,0.994989,0.122549,0.985714,0.084731
2,Cue CE,70,0.056145,0.985046,0.928900,0.985361,0.156663,0.971429,0.084731
0,Clean CE,70,0.955530,0.812917,-0.142613,-0.032819,0.196655,0.157143,0.084731
